# Expérimentations SOM (Self-Organizing Map)

Ce notebook conduit les expérimentations de la carte de Kohonen selon les axes : **hyperparamètres**, **convergence**, **visualisation topologique**, **compression** et **génération**. Chaque expérience SOM est répétée sur 2 seeds pour contrôler la stochasticité.

## 0. Setup expérimental

Mêmes conventions que le notebook PCA : seed, split, prétraitement.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("__file__"), "..")))

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from collections import defaultdict
import json
import time

from src.som import SOM
from src.dataset import load_mnist_dataset
from src.helper import extract_full_dataset
from src.metrics import compression_report

In [ ]:
SEED = 42
SEEDS = [42, 123]  # 2 seeds pour la répétition des expériences stochastiques
rng = np.random.default_rng(SEED)

dataloader = load_mnist_dataset(train=True, batch_size=4096, download=True, shuffle=False)
images, digit_labels = extract_full_dataset(dataloader)

N = 10000
X_all = images[:N].flatten(start_dim=1).numpy()
labels_all = digit_labels[:N].numpy()

indices = rng.permutation(N)
split = int(0.8 * N)
train_idx, test_idx = indices[:split], indices[split:]

X_train, X_test = X_all[train_idx], X_all[test_idx]
y_train, y_test = labels_all[train_idx], labels_all[test_idx]

print(f"X_train : {X_train.shape} | X_test : {X_test.shape}")
print(f"Seeds : {SEEDS}")

## 0bis. Visualisation jouet 2D : la grille qui épouse les données

Avant MNIST (784D), on vérifie visuellement sur un problème 2D que la SOM se déplie correctement et que la topologie de la grille est préservée.

In [ ]:
rng_toy = np.random.default_rng(42)
X_toy = rng_toy.uniform(-1.0, 1.0, (1000, 2))

grid_shape_toy = (5, 5)
som_toy = SOM(grid_shape=grid_shape_toy, alpha=0.6, gamma=1.5,
              n_iterations=3000, random_state=42)
som_toy.fit(X_toy)

H_toy, W_toy = grid_shape_toy
W_grid = som_toy.W.reshape(H_toy, W_toy, 2)

fig = go.Figure()

# Points de données
fig.add_trace(go.Scatter(
    x=X_toy[:, 0], y=X_toy[:, 1], mode="markers",
    marker=dict(size=4, color="rgba(180,180,180,0.5)"), name="Data"
))

# Lignes horizontales de la grille
for r in range(H_toy):
    fig.add_trace(go.Scatter(
        x=W_grid[r, :, 0], y=W_grid[r, :, 1], mode="lines+markers",
        line=dict(color="crimson", width=2), marker=dict(size=6),
        showlegend=(r == 0), name="Grid"
    ))

# Lignes verticales de la grille
for c in range(W_toy):
    fig.add_trace(go.Scatter(
        x=W_grid[:, c, 0], y=W_grid[:, c, 1], mode="lines",
        line=dict(color="crimson", width=2), showlegend=False
    ))

fig.update_layout(width=600, height=600, title="SOM 5x5 sur données uniformes 2D",
                  xaxis_title="x", yaxis_title="y")
fig.show()

## 1. Grille d'hyperparamètres

**Hypothèse** : La décroissance linéaire (schedule classique de Kohonen) donne de meilleurs résultats qu'un taux fixe, car elle permet une organisation globale au début (grand rayon/pas) puis un affinement local.

**Protocole** : Grille systématique sur alpha, gamma, taille de grille et mode de décroissance. Chaque combinaison est répétée sur 2 seeds, avec report de la moyenne et de l'écart-type.

**Métriques suivies** :
- Erreur de quantification (QE) : distance moyenne échantillon-BMU
- Erreur topographique (TE) : proportion dont le 2e BMU n'est pas voisin direct du 1er
- MSE de reconstruction (via `compression_report`)

In [ ]:
import itertools

alpha_values = [0.1, 0.5]
gamma_values = [1.0, 3.0]
grid_sizes = [(5, 5), (8, 8), (10, 10), (15, 15)]
decay_modes = ["linear", "none"]
N_ITER_GRID = 10000

total_runs = len(alpha_values) * len(gamma_values) * len(grid_sizes) * len(decay_modes) * len(SEEDS)
print(f"Total d'expériences : {total_runs}")

grid_results = []
t0 = time.time()

for i, (alpha, gamma, grid, decay, seed) in enumerate(
    itertools.product(alpha_values, gamma_values, grid_sizes, decay_modes, SEEDS)
):
    elapsed = time.time() - t0
    print(f"\r[{i + 1}/{total_runs}] grid={grid[0]}x{grid[1]}, "
          f"alpha={alpha}, gamma={gamma}, decay={decay}, seed={seed} "
          f"({elapsed:.0f}s)", end="", flush=True)

    som = SOM(grid_shape=grid, alpha=alpha, gamma=gamma,
              n_iterations=N_ITER_GRID, decay=decay, random_state=seed)
    som.fit(X_train)

    qe = som.quantization_error(X_test)
    te = som.topographic_error(X_test)

    latent = som.encode(X_test)
    X_rec = som.decode(latent)
    report = compression_report(som.get_codebook(), latent, X_test, X_rec)

    grid_results.append({
        "alpha": alpha, "gamma": gamma,
        "grid": f"{grid[0]}x{grid[1]}", "n_neurons": grid[0] * grid[1],
        "decay": decay, "seed": seed,
        "quantization_error": qe,
        "topographic_error": te,
        "mse": report["reconstruction_mse"],
        "total_bytes": report["total_compressed_bytes"],
        "compression_ratio": report["compression_ratio"],
    })

print(f"\nTerminé en {time.time() - t0:.0f}s")

In [ ]:
# Agrégation par configuration (moyenne et écart-type sur les seeds)
grouped = defaultdict(list)
for r in grid_results:
    key = (r["alpha"], r["gamma"], r["grid"], r["decay"])
    grouped[key].append(r)

print(f"{'Grid':>5} | {'a':>4} | {'g':>4} | {'Decay':>7} | "
      f"{'QE (m +/- s)':>18} | {'TE (m +/- s)':>18} | {'MSE (m +/- s)':>22}")
print("-" * 105)

aggregated = []
for key in sorted(grouped.keys()):
    runs = grouped[key]
    alpha, gamma, grid, decay = key

    qe_m = np.mean([r["quantization_error"] for r in runs])
    qe_s = np.std([r["quantization_error"] for r in runs])
    te_m = np.mean([r["topographic_error"] for r in runs])
    te_s = np.std([r["topographic_error"] for r in runs])
    mse_m = np.mean([r["mse"] for r in runs])
    mse_s = np.std([r["mse"] for r in runs])

    aggregated.append({"alpha": alpha, "gamma": gamma, "grid": grid, "decay": decay,
                       "qe_mean": qe_m, "qe_std": qe_s,
                       "te_mean": te_m, "te_std": te_s,
                       "mse_mean": mse_m, "mse_std": mse_s})

    print(f"{grid:>5} | {alpha:>4.1f} | {gamma:>4.1f} | {decay:>7} | "
          f"{qe_m:>7.4f} +/- {qe_s:>6.4f} | {te_m:>7.4f} +/- {te_s:>6.4f} | "
          f"{mse_m:>8.6f} +/- {mse_s:>10.6f}")

# Identifier la meilleure config (MSE minimale)
best = min(aggregated, key=lambda x: x["mse_mean"])
print(f"\nMeilleure config (MSE) : grid={best['grid']}, alpha={best['alpha']}, "
      f"gamma={best['gamma']}, decay={best['decay']}")

In [ ]:
# Heatmap QE vs TE pour visualiser le compromis
# Filtrer : decay=linear seulement pour la heatmap
linear_agg = [a for a in aggregated if a["decay"] == "linear"]

fig = go.Figure()
for a in linear_agg:
    fig.add_trace(go.Scatter(
        x=[a["qe_mean"]], y=[a["te_mean"]],
        mode="markers+text",
        text=[f"{a['grid']} a={a['alpha']} g={a['gamma']}"],
        textposition="top center",
        marker=dict(size=12, color=a["mse_mean"], colorscale="Viridis",
                    showscale=True, colorbar=dict(title="MSE")),
        showlegend=False
    ))

fig.update_layout(
    title="Compromis Quantization Error vs Topographic Error (decay=linear)",
    xaxis_title="Quantization Error (QE) - plus bas = meilleur",
    yaxis_title="Topographic Error (TE) - plus bas = meilleur",
    width=900, height=600
)
fig.show()

**Conclusion (1)** : *A compléter après exécution.* Vérifier que (1) decay=linear surpasse decay=none, (2) QE et TE évoluent en tension (grille plus grande = meilleur QE mais TE potentiellement plus fragile), (3) identifier le compromis optimal.

## 2. Analyse de convergence

**Hypothèse** : La décroissance linéaire produit une convergence plus stable et un meilleur résultat final que le taux fixe.

**Protocole** : Entraîner la meilleure taille de grille avec decay=linear vs decay=none, tracer la quantization error au cours de l'entraînement.

In [ ]:
# Utiliser la meilleure config trouvée en section 1
best_grid = tuple(int(x) for x in best["grid"].split("x"))
best_alpha = best["alpha"]
best_gamma = best["gamma"]

fig = go.Figure()

for decay_mode in ["linear", "none"]:
    for seed in SEEDS:
        som_conv = SOM(grid_shape=best_grid, alpha=best_alpha, gamma=best_gamma,
                       n_iterations=N_ITER_GRID, decay=decay_mode, random_state=seed)
        som_conv.fit(X_train)

        # history_ contient des échantillons réguliers de la QE pendant l'entraînement
        x_axis = np.linspace(0, N_ITER_GRID, len(som_conv.history_)).tolist()
        fig.add_trace(go.Scatter(
            x=x_axis, y=som_conv.history_,
            mode="lines", name=f"{decay_mode} (seed={seed})",
            opacity=0.8
        ))

fig.update_layout(
    title=f"Convergence SOM {best['grid']} (alpha={best_alpha}, gamma={best_gamma})",
    xaxis_title="Itération",
    yaxis_title="Quantization Error (échantillonnée)",
    width=900, height=500
)
fig.show()

**Conclusion (2)** : *A compléter après exécution.* La décroissance linéaire devrait montrer une convergence plus rapide et un plateau final plus bas, confirmant l'hypothèse du schedule de Kohonen.

## 3. Visualisation topologique

### 3.1 U-Matrix et grille annotée

**Hypothèse** : La carte topologique doit regrouper spatialement les classes visuellement similaires (4/9, 3/8, 5/6). La U-Matrix doit révéler des "frontières" (zones de haute distance) entre clusters de chiffres différents.

**Protocole** : Entraîner une SOM avec la meilleure config, calculer la U-Matrix, afficher la grille des prototypes + classe majoritaire + pureté.

In [ ]:
# Entraîner la SOM de référence
som_best = SOM(grid_shape=best_grid, alpha=best_alpha, gamma=best_gamma,
               n_iterations=N_ITER_GRID, decay="linear", random_state=42)
som_best.fit(X_train)

latent_best = som_best.encode(X_test)

print(f"QE = {som_best.quantization_error(X_test):.4f}")
print(f"TE = {som_best.topographic_error(X_test):.4f}")

In [ ]:
# Calcul de la U-Matrix
def compute_u_matrix(som):
    """Compute the unified distance matrix (mean distance to direct neighbors)."""
    H, W_g = som.grid_shape
    u_mat = np.zeros((H, W_g))
    for r in range(H):
        for c in range(W_g):
            idx = r * W_g + c
            neighbors = []
            if r > 0: neighbors.append((r - 1) * W_g + c)
            if r < H - 1: neighbors.append((r + 1) * W_g + c)
            if c > 0: neighbors.append(r * W_g + c - 1)
            if c < W_g - 1: neighbors.append(r * W_g + c + 1)
            dists = [np.linalg.norm(som.W[idx] - som.W[n]) for n in neighbors]
            u_mat[r, c] = np.mean(dists)
    return u_mat

u_matrix = compute_u_matrix(som_best)

fig = go.Figure(data=go.Heatmap(
    z=u_matrix[::-1], colorscale="Hot", showscale=True,
    colorbar=dict(title="Distance")
))
fig.update_layout(
    title="U-Matrix : distances inter-neurones voisins",
    width=600, height=600, xaxis_title="Col", yaxis_title="Row"
)
fig.show()

In [ ]:
H_best, W_best = best_grid

# Grille des prototypes
fig = make_subplots(rows=H_best, cols=W_best,
                    horizontal_spacing=0.005, vertical_spacing=0.005)

for r in range(H_best):
    for c in range(W_best):
        idx = r * W_best + c
        neuron_img = som_best.W[idx].reshape(28, 28)
        fig.add_trace(
            go.Heatmap(z=neuron_img[::-1], colorscale="gray", showscale=False),
            row=r + 1, col=c + 1
        )
        fig.update_xaxes(visible=False, row=r + 1, col=c + 1)
        fig.update_yaxes(visible=False, row=r + 1, col=c + 1)

fig.update_layout(height=80 * H_best, width=80 * W_best,
                  title_text="Prototypes des neurones (grille topologique)")
fig.show()

In [ ]:
# Classe majoritaire et pureté par neurone
class_map = np.full((H_best, W_best), -1, dtype=int)
purity_map = np.zeros((H_best, W_best))

latent_train_best = som_best.encode(X_train)

for n_idx in range(som_best.n_neurons):
    mask = latent_train_best.array == n_idx
    if np.sum(mask) == 0:
        continue
    labels_n = y_train[mask]
    counts = np.bincount(labels_n, minlength=10)
    class_map[n_idx // W_best, n_idx % W_best] = int(np.argmax(counts))
    purity_map[n_idx // W_best, n_idx % W_best] = counts.max() / counts.sum()

fig = make_subplots(rows=1, cols=2, subplot_titles=["Classe majoritaire", "Pureté"])

fig.add_trace(
    go.Heatmap(z=class_map[::-1], colorscale="Turbo", showscale=True,
               colorbar=dict(title="Classe", x=0.45)),
    row=1, col=1
)
fig.add_trace(
    go.Heatmap(z=purity_map[::-1], colorscale="Greens", showscale=True,
               colorbar=dict(title="Pureté")),
    row=1, col=2
)

fig.update_layout(height=500, width=1000, title_text="Cartographie des classes et pureté")
for c in range(1, 3):
    fig.update_xaxes(title_text="Col", row=1, col=c)
    fig.update_yaxes(title_text="Row", row=1, col=c)
fig.show()

global_purity = np.mean(purity_map[purity_map > 0])
print(f"Pureté globale moyenne : {global_purity:.4f}")

**Conclusion (3)** : *A compléter après exécution.* Vérifier que les chiffres visuellement similaires (4/9, 3/8) se retrouvent spatialement proches sur la carte. La U-Matrix devrait montrer des zones de forte distance entre les régions dominées par des classes différentes.

## 4. Compression et courbe Rate-Distortion

**Hypothèse** : A budget d'octets équivalent, la SOM devrait faire moins bien que K-Means en pure MSE (car la contrainte topologique restreint les prototypes) mais offre en échange l'interprétabilité spatiale. C'est un vrai trade-off, pas un "SOM est moins bon".

**Protocole** : Pour chaque taille de grille, entraîner avec les meilleurs hyperparamètres, calculer le couple (bytes, MSE) sur le test set.

In [ ]:
rd_grid_sizes = [(3, 3), (4, 4), (5, 5), (6, 6), (8, 8), (10, 10), (12, 12), (15, 15)]
N_ITER_RD = 10000

som_rd_results = []

for grid in rd_grid_sizes:
    seed_results = []
    for seed in SEEDS:
        som_rd = SOM(grid_shape=grid, alpha=best_alpha, gamma=best_gamma,
                     n_iterations=N_ITER_RD, decay="linear", random_state=seed)
        som_rd.fit(X_train)

        latent_rd = som_rd.encode(X_test)
        X_rec_rd = som_rd.decode(latent_rd)
        report = compression_report(som_rd.get_codebook(), latent_rd, X_test, X_rec_rd)

        seed_results.append(report)

    mse_m = np.mean([r["reconstruction_mse"] for r in seed_results])
    mse_s = np.std([r["reconstruction_mse"] for r in seed_results])
    # Les bytes sont déterministes (dépendent que de la taille de grille)
    total_b = seed_results[0]["total_compressed_bytes"]

    som_rd_results.append({
        "grid": f"{grid[0]}x{grid[1]}",
        "n_neurons": grid[0] * grid[1],
        "total_bytes": total_b,
        "mse_mean": mse_m,
        "mse_std": mse_s,
        "codebook_bytes": seed_results[0]["codebook_bytes"],
        "latent_bytes": seed_results[0]["latent_bytes"],
    })

    print(f"Grid {grid[0]:>2}x{grid[1]:<2} | "
          f"MSE = {mse_m:.6f} +/- {mse_s:.6f} | "
          f"Bytes = {total_b:>10d}")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=[r["total_bytes"] for r in som_rd_results],
    y=[r["mse_mean"] for r in som_rd_results],
    error_y=dict(type="data", array=[r["mse_std"] for r in som_rd_results], visible=True),
    mode="lines+markers+text",
    text=[r["grid"] for r in som_rd_results],
    textposition="top center",
    name="SOM",
    marker=dict(size=8)
))
fig.update_layout(
    title="Courbe Rate-Distortion SOM (test set, mean +/- std sur 2 seeds)",
    xaxis_title="Taille totale (octets)",
    yaxis_title="MSE de reconstruction",
    xaxis_type="log",
    width=900, height=500
)
fig.show()

**Conclusion (4)** : *A compléter après exécution.* Commenter la forme de la courbe et le rendement décroissant en ajoutant des neurones.

## 5. Génération de données synthétiques

### 5.1 Décodage de neurones

**Hypothèse** : La génération SOM est fondamentalement limitée aux K prototypes appris. Un neurone "vide" (jamais gagnant) produit un prototype interpolé par la contrainte topologique, pas un artefact aléatoire.

**Protocole A** : Décoder des neurones choisis au hasard. Décoder un neurone vide si existant.

In [ ]:
# Neurones gagnants et vides
winning_counts = np.bincount(latent_best.array, minlength=som_best.n_neurons)
active_neurons = np.where(winning_counts > 0)[0]
empty_neurons = np.where(winning_counts == 0)[0]

print(f"Neurones actifs : {len(active_neurons)} / {som_best.n_neurons}")
print(f"Neurones vides : {len(empty_neurons)}")

# Afficher quelques neurones au hasard + les vides
rng_show = np.random.default_rng(SEED)
show_active = rng_show.choice(active_neurons, size=min(8, len(active_neurons)), replace=False)
show_empty = empty_neurons[:min(4, len(empty_neurons))] if len(empty_neurons) > 0 else []

all_show = list(show_active) + list(show_empty)
n_show = len(all_show)

if n_show > 0:
    cols = min(n_show, 8)
    rows = (n_show + cols - 1) // cols
    titles = [f"N{idx} ({'vide' if idx in empty_neurons else f'{winning_counts[idx]} hits'})" for idx in all_show]

    fig = make_subplots(rows=rows, cols=cols, subplot_titles=titles)
    for i, idx in enumerate(all_show):
        r, c = i // cols + 1, i % cols + 1
        fig.add_trace(
            go.Heatmap(z=som_best.W[idx].reshape(28, 28)[::-1], colorscale="gray", showscale=False),
            row=r, col=c
        )
        fig.update_xaxes(visible=False, row=r, col=c)
        fig.update_yaxes(visible=False, row=r, col=c)

    fig.update_layout(height=200 * rows, width=120 * cols,
                      title_text="Prototypes : neurones actifs vs vides")
    fig.show()

### 5.2 Interpolation topologique

**Protocole B** : Se déplacer le long d'une ligne sur la grille et décoder chaque neurone traversé.

**Hypothèse** : La contrainte topologique produit des transitions plus douces et sémantiquement cohérentes que l'interpolation gaussienne naïve de la PCA.

In [ ]:
H_b, W_b = best_grid

# Traversée horizontale de la première ligne
path_indices = [0 * W_b + c for c in range(W_b)]
path_images = som_best.W[path_indices]

fig = make_subplots(rows=1, cols=len(path_indices),
                    subplot_titles=[f"({0},{c})" for c in range(W_b)])
for i, idx in enumerate(path_indices):
    fig.add_trace(
        go.Heatmap(z=path_images[i].reshape(28, 28)[::-1], colorscale="gray", showscale=False),
        row=1, col=i + 1
    )
    fig.update_xaxes(visible=False, row=1, col=i + 1)
    fig.update_yaxes(visible=False, row=1, col=i + 1)
fig.update_layout(height=200, width=100 * len(path_indices),
                  title_text="Interpolation topologique : traversée horizontale (ligne 0)")
fig.show()

# Traversée verticale de la première colonne
path_v = [r * W_b + 0 for r in range(H_b)]
path_v_images = som_best.W[path_v]

fig2 = make_subplots(rows=1, cols=len(path_v),
                     subplot_titles=[f"({r},0)" for r in range(H_b)])
for i, idx in enumerate(path_v):
    fig2.add_trace(
        go.Heatmap(z=path_v_images[i].reshape(28, 28)[::-1], colorscale="gray", showscale=False),
        row=1, col=i + 1
    )
    fig2.update_xaxes(visible=False, row=1, col=i + 1)
    fig2.update_yaxes(visible=False, row=1, col=i + 1)
fig2.update_layout(height=200, width=100 * len(path_v),
                   title_text="Interpolation topologique : traversée verticale (colonne 0)")
fig2.show()

### 5.3 Slider interactif

Widget interactif pour explorer la grille SOM en temps réel : chaque déplacement du slider affiche le prototype du neurone correspondant.

In [ ]:
from ipywidgets import interact, IntSlider
from IPython.display import display

def show_neuron(row, col):
    """Display the prototype image of the neuron at position (row, col)."""
    idx = row * som_best.grid_shape[1] + col
    img = som_best.W[idx].reshape(28, 28)
    majority_class = class_map[row, col]
    purity = purity_map[row, col]

    plt.figure(figsize=(4, 4))
    plt.imshow(img, cmap="gray")
    plt.title(f"Neurone ({row}, {col}) | Classe maj. = {majority_class} | Pureté = {purity:.2f}")
    plt.axis("off")
    plt.show()

interact(
    show_neuron,
    row=IntSlider(min=0, max=best_grid[0] - 1, step=1, value=0, description="Row"),
    col=IntSlider(min=0, max=best_grid[1] - 1, step=1, value=0, description="Col"),
)

**Conclusion (5)** : *A compléter après exécution.* La génération SOM est limitée aux K prototypes appris (pas d'interpolation continue). Mais la topologie organise les transitions de manière interprétable : les neurones voisins se ressemblent visuellement. Le slider permet de vérifier ce que chaque "axe" de la carte encode (ex. épaisseur, inclinaison, forme).

## 6. Sauvegarde des résultats

In [ ]:
som_rd = {
    "method": "SOM",
    "points": [
        {
            "label": r["grid"],
            "total_bytes": int(r["total_bytes"]),
            "mse": float(r["mse_mean"]),
            "mse_std": float(r["mse_std"]),
            "codebook_bytes": int(r["codebook_bytes"]),
            "latent_bytes": int(r["latent_bytes"]),
        }
        for r in som_rd_results
    ],
}

os.makedirs("data", exist_ok=True)
with open("data/som_rate_distortion.json", "w") as f:
    json.dump(som_rd, f, indent=2)

print("Résultats sauvegardés dans data/som_rate_distortion.json")